# Exploratory Data Analysis

Subject: Titanic Survival Prediction

## Overview

In [1]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
import polars as pl

PASTEL = px.colors.qualitative.Pastel
SURV_COLORS = {"0": PASTEL[1], "1": PASTEL[0]}


In [2]:
raw = pl.read_parquet("data/train.parquet")
raw.head()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


In [3]:
df = raw.clone()
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")
df.schema

Shape: (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


Schema([('PassengerId', Int64),
        ('Survived', Int64),
        ('Pclass', Int64),
        ('Name', String),
        ('Sex', String),
        ('Age', Float64),
        ('SibSp', Int64),
        ('Parch', Int64),
        ('Ticket', String),
        ('Fare', Float64),
        ('Cabin', String),
        ('Embarked', String)])

In [4]:
# Missing values per column
df.null_count()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,177,0,0,0,0,687,2


In [5]:
nulls = df.null_count().unpivot().filter(pl.col("value") > 0)
fig = px.bar(
    nulls, x="variable", y="value",
    color_discrete_sequence=[PASTEL[4]],
    labels={"variable": "Column", "value": "Missing count"},
    title="Missing Values per Column",
    text_auto=True,
)
fig.update_layout(showlegend=False, width=600, height=400)
fig.show()


In [6]:
# Drop columns, drop rows with null Age/Embarked
df = df.drop(["PassengerId", "Name", "Ticket", "Cabin"])
df = df.drop_nulls(subset=["Age", "Embarked"])
df.null_count()

Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0


In [7]:
# Create age category and Age_Sex interaction feature
df = df.with_columns(
    pl.when(pl.col("Age") < 18)
    .then(pl.lit("Minor"))
    .otherwise(pl.lit("Adult"))
    .alias("Age_label"),
)
df = df.with_columns(
    (pl.col("Age_label") + " " + pl.col("Sex")).alias("Age_Sex")
)
df.select(["Pclass", "Sex", "Age", "Age_label", "Age_Sex", "Survived"]).head()

Pclass,Sex,Age,Age_label,Age_Sex,Survived
i64,str,f64,str,str,i64
3,"""male""",22.0,"""Adult""","""Adult male""",0
1,"""female""",38.0,"""Adult""","""Adult female""",1
3,"""female""",26.0,"""Adult""","""Adult female""",1
1,"""female""",35.0,"""Adult""","""Adult female""",1
3,"""male""",35.0,"""Adult""","""Adult male""",0


## Exploring the Data

Observation of distribution by categories and correlation between categories.

In [8]:
# Distribution overview — visual instead of print statements
cats = ["Pclass", "Sex", "Age_label", "Survived", "Embarked"]
fig = sp.make_subplots(rows=1, cols=len(cats), subplot_titles=cats)
for i, col in enumerate(cats, 1):
    counts = df[col].value_counts().sort(col)
    fig.add_trace(
        go.Bar(
            x=counts[col].cast(pl.String).to_list(),
            y=counts["count"].to_list(),
            marker_color=PASTEL[:len(counts)],
            text=counts["count"].to_list(),
            textposition="outside",
            showlegend=False,
        ),
        row=1, col=i,
    )
fig.update_layout(title="Category Distributions", height=400, width=1200)
fig.show()

In [9]:
numeric_cols = [c for c in df.columns if df[c].dtype in (pl.Float64, pl.Int64)]
corr = np.corrcoef(df.select(numeric_cols).to_numpy().T)
fig = px.imshow(
    corr, x=numeric_cols, y=numeric_cols,
    text_auto=".2f", color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1, title="Correlation Matrix",
)
fig.update_layout(width=700, height=600)
fig.show()


#### Does class and gender influence survival chances?

In [10]:
def survival_bars(data, x, hue, title, width=900, height=400):
    """Side-by-side subplots: mean survival rate + survivor count."""
    fig = sp.make_subplots(rows=1, cols=2, subplot_titles=["Survival Rate", "Survivor Count"])
    rate = data.group_by([x, hue]).agg(pl.col("Survived").mean().alias("rate")).sort([x, hue])
    count = data.group_by([x, hue]).agg(pl.col("Survived").sum().alias("count")).sort([x, hue])
    for i, hv in enumerate(sorted(data[hue].unique().to_list())):
        r = rate.filter(pl.col(hue) == hv)
        c = count.filter(pl.col(hue) == hv)
        fig.add_trace(go.Bar(
            x=r[x].cast(pl.String).to_list(), y=r["rate"].to_list(),
            name=str(hv), marker_color=PASTEL[i], showlegend=True,
            text=[f"{v:.0%}" for v in r["rate"].to_list()], textposition="outside",
        ), row=1, col=1)
        fig.add_trace(go.Bar(
            x=c[x].cast(pl.String).to_list(), y=c["count"].to_list(),
            name=str(hv), marker_color=PASTEL[i], showlegend=False,
            text=c["count"].to_list(), textposition="outside",
        ), row=1, col=2)
    fig.update_layout(title=title, barmode="group", height=height, width=width)
    return fig

survival_bars(df, x="Sex", hue="Pclass", title="Survival by Sex and Class").show()


In [11]:
survival_bars(df, x="Pclass", hue="Sex", title="Survival by Class and Sex").show()

Both class and gender appear to be highly discriminating factors in measuring passenger survival rates.
If we ranked people by survival likelihood, the results speak for themselves:
1. Woman in 1st class
2. Woman in 2nd class
3. Woman in 3rd class
4. Man in 1st class
5. Man in 2nd class
6. Man in 3rd class

We will explore the rest of the data to see if there are other discriminating factors in the survival rate.

#### Does the port of embarkation influence survival chances?

In [12]:
emb = df.group_by("Embarked").agg(
    pl.col("Survived").mean().alias("rate"),
    pl.col("Survived").sum().alias("survivors"),
    pl.len().alias("total"),
).sort("Embarked")
fig = px.bar(
    emb, x="Embarked", y="rate",
    text_auto=".0%", color="Embarked", color_discrete_sequence=PASTEL,
    title="Survival Rate by Embarkation Port",
    labels={"rate": "Survival Rate"},
)
fig.update_layout(width=500, height=400, showlegend=False)
fig.show()


In [13]:
emb_class = (
    df.group_by(["Embarked", "Pclass"])
    .agg(pl.len().alias("count"))
    .with_columns(pl.col("Pclass").cast(pl.String))
    .sort(["Embarked", "Pclass"])
)
fig = px.bar(
    emb_class, x="Embarked", y="count", color="Pclass",
    barmode="group", color_discrete_sequence=PASTEL,
    title="Class Distribution per Embarkation Port",
    text_auto=True,
)
fig.update_layout(width=600, height=400)
fig.show()


Several interesting data points should be noted:
- The embarkation port "Q" for Queenstown in Ireland was primarily used by passengers traveling in 3rd class. Only 2 people were in 1st, 2 in 2nd class versus 24 in 3rd.
- The embarkation port "C" for Cherbourg in France was primarily used by passengers traveling in 1st class: 74 people versus only 15 for 2nd class and 41 for 3rd class. The "C" category is thus more represented by 1st class passengers while for "Q" and "S" the 1st class is often the category with the fewest passengers.

In [14]:
survival_bars(df, x="Embarked", hue="Sex", title="Survival by Embarkation Port and Sex").show()

In [15]:
survival_bars(df, x="Embarked", hue="Pclass", title="Survival by Embarkation Port and Class").show()

We find the same pattern: women survived more than men, regardless of embarkation port.
The difference between the 3 categories lies more in the class distribution:
- There are more 1st class passengers for Cherbourg
- There are almost exclusively 3rd class passengers for Queenstown.

For "Q":
- 1st and 2nd class, only 2 people per class and a 50% survival rate per class. The average is not relevant for so few data points.

For "C":
- The 3rd class survival rate is 43%, higher than "Q" and "S", which are 25% and 21% respectively. It remains to be seen if this is justified by a larger proportion of women in the "C" group.

In [16]:
emb_sex = df.group_by(["Embarked", "Sex"]).agg(pl.len().alias("count")).sort(["Embarked", "Sex"])
fig = px.bar(
    emb_sex, x="Embarked", y="count", color="Sex",
    barmode="group", color_discrete_sequence=PASTEL,
    title="Gender Distribution per Embarkation Port",
    text_auto=True,
)
fig.update_layout(width=600, height=400)
fig.show()


The distribution by gender outside of class is fairly homogeneous for "Q" and "C", but heterogeneous for "S". There are far more men than women, which may help explain the lower survival rate for those embarking at Southampton compared to Cherbourg or Queenstown.

The survival rate thus appears to be more linked to two criteria:
- the class of the passenger
- the gender

Are there other discriminating factors?

#### Does age and the adult/minor distinction influence survival chances?

In [17]:
fig = px.histogram(
    df.with_columns(pl.col("Survived").cast(pl.String)),
    x="Age", nbins=30, color="Survived",
    color_discrete_map=SURV_COLORS,
    barmode="overlay", opacity=0.7,
    title="Age Distribution by Survival",
    category_orders={"Survived": ["0", "1"]},
)
fig.update_layout(width=800, height=400)
fig.show()


In [18]:
survival_bars(df, x="Age_label", hue="Sex", title="Survival by Age Group and Sex").show()

In [19]:
survival_bars(df, x="Age_label", hue="Pclass", title="Survival by Age Group and Class").show()

In [20]:
order = ["Minor male", "Minor female", "Adult male", "Adult female"]
asx = df.group_by(["Age_Sex", "Pclass"]).agg(
    pl.col("Survived").mean().alias("rate"),
    pl.len().alias("count"),
)
fig = sp.make_subplots(rows=1, cols=2, subplot_titles=["Survival Rate", "Passenger Count"])
for pc in sorted(asx["Pclass"].unique().to_list()):
    sub = (
        asx.filter(pl.col("Pclass") == pc)
        .with_columns(pl.col("Age_Sex").cast(pl.Enum(order)))
        .sort("Age_Sex")
    )
    fig.add_trace(go.Bar(
        x=sub["Age_Sex"].cast(pl.String).to_list(), y=sub["rate"].to_list(),
        name=f"Class {pc}", marker_color=PASTEL[pc - 1],
        text=[f"{v:.0%}" for v in sub["rate"].to_list()], textposition="outside",
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=sub["Age_Sex"].cast(pl.String).to_list(), y=sub["count"].to_list(),
        name=f"Class {pc}", marker_color=PASTEL[pc - 1],
        showlegend=False, text=sub["count"].to_list(), textposition="outside",
    ), row=1, col=2)
fig.update_layout(title="Age_Sex × Class Interaction", barmode="group", height=450, width=1000)
fig.show()


The minor/adult criterion reduces the gap in survival rate between boys and girls, but the survival rate is still higher for girls (~70%) than for boys (~39%). Minors have a higher survival rate than adults in any case. However, note that the sample size for minors is smaller than for adults.

This criterion therefore remains discriminatory and can be added to class and gender to study a passenger's survival rate.

### Do survival chances increase when paying a higher ticket price?

In [21]:
fig = px.histogram(
    df.with_columns(pl.col("Survived").cast(pl.String)),
    x="Fare", nbins=50, color="Survived",
    color_discrete_map=SURV_COLORS,
    barmode="overlay", opacity=0.7,
    title="Fare Distribution by Survival",
    category_orders={"Survived": ["0", "1"]},
)
fig.update_layout(width=800, height=400)
fig.show()


In [22]:
fare_agg = (
    df.group_by(["Pclass", "Survived"])
    .agg(pl.col("Fare").mean().alias("mean_fare"))
    .with_columns(pl.col("Survived").cast(pl.String))
    .sort(["Pclass", "Survived"])
)
fig = px.bar(
    fare_agg, x="Pclass", y="mean_fare", color="Survived",
    barmode="group", color_discrete_map=SURV_COLORS,
    title="Mean Fare by Class and Survival",
    text_auto=".0f",
)
fig.update_layout(width=600, height=400)
fig.show()


In [23]:
fig = px.scatter(
    df.with_columns(pl.col("Survived").cast(pl.String)),
    x="Age", y="Fare", color="Survived",
    color_discrete_map=SURV_COLORS,
    opacity=0.6, title="Fare vs Age by Survival",
    hover_data=["Sex", "Pclass"],
    category_orders={"Survived": ["0", "1"]},
)
fig.update_layout(width=800, height=500)
fig.show()


A high ticket price appears to increase survival chances but may be linked to the departure port and passenger class.